In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [2]:
real = pd.read_csv("True.csv")
fake = pd.read_csv("Fake.csv")

real["label"] = 1
fake["label"] = 0

data = pd.concat([real, fake], ignore_index=True)

data["text"] = data["text"].fillna("")

data["text_length"] = data["text"].str.split().str.len()

data.head()

,title,text,subject,date,label,text_length
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",1,749
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",1,624
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",1,457
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",1,376
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",1,852


In [3]:
# Divide Articles into Different Text Length Groups
short_limit = data["text_length"].quantile(0.25)
long_limit = data["text_length"].quantile(0.75)

data["length_group"] = pd.cut(
    data["text_length"],
    bins=[-1, short_limit, long_limit, float("inf")],
    labels=["Short", "Medium", "Long"]
)

data["length_group"].value_counts()

length_group
Medium    22472
Short     11238
Long      11188
Name: count, dtype: int64

In [4]:
# Split Dataset
X_train, X_test, y_train, y_test = train_test_split(
    data["text"], data["label"],
    test_size=0.2, random_state=42, stratify=data["label"]
)

test_groups = data.loc[X_test.index, "length_group"]

In [5]:
# Train Logistic Regression Model
model = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000)),
    ("lr", LogisticRegression())
])

model.fit(X_train, y_train)

Pipeline(steps=[('tfidf', TfidfVectorizer(max_features=5000)),
                ('lr', LogisticRegression())])

In [6]:
# Evaluate Model on Different Text Length Groups
results = []

for group in ["Short", "Medium", "Long"]:

    X_group = X_test[test_groups == group]
    y_group = y_test[test_groups == group]

    y_pred = model.predict(X_group)

    results.append({
        "Text Length": group,
        "Number of Articles": len(X_group),
        "Accuracy": accuracy_score(y_group, y_pred),
        "Precision": precision_score(y_group, y_pred),
        "Recall": recall_score(y_group, y_pred),
        "F1 Score": f1_score(y_group, y_pred)
    })

results = pd.DataFrame(results)

results

,Text Length,Number of Articles,Accuracy,Precision,Recall,F1 Score
0,Short,2232,0.995968,0.995287,0.997638,0.996461
1,Medium,4507,0.986465,0.980188,0.987914,0.984036
2,Long,2241,0.979473,0.976723,0.981998,0.979354
